# 01 · Problem Setup — Why Does a Law Firm Care About This?

> *"We want our paralegals to use AI to summarise client matters. But we can't put confidential matter data into a public model. Can we just strip out the names first?"*

That question — paraphrased from a real conversation, but the version a hundred firms are having right now — is the entry point for this whole project.

This notebook lays out **why naive name-stripping isn't enough**, using the [Text Anonymization Benchmark (TAB)](https://github.com/NorskRegnesentral/text-anonymization-benchmark) as our empirical anchor.

By the end of this notebook you'll see:

1. What kinds of identifying information actually appear in legal documents (it's much more than names).
2. The distinction TAB draws between **DIRECT**, **QUASI**, and **NO_MASK** identifiers — and why QUASI is where the real risk lives.
3. A first taste of the **mosaic effect**: how harmless-looking facts combine into a unique fingerprint.

Notebooks `02` and `03` then run real models against the data and dig into the mosaic effect properly.

---


## The Dataset

**TAB (Text Anonymization Benchmark)** is 1,268 court cases from the European Court of Human Rights, manually annotated by legal NLP researchers at the [Norwegian Computing Center](https://github.com/NorskRegnesentral/text-anonymization-benchmark).

Why ECHR cases for a law-firm-themed project?

- They are **real legal text** with the same structural features as the matter notes a UK firm would handle: factual narrative, named parties, dates, references to other cases.
- They are **already public** (court rulings are published), which means we can experiment without exposing real client data.
- Most importantly, every entity has been hand-labelled with three pieces of metadata: its type (`PERSON`, `LOC`, `CODE`, …), its **identifier role** (`DIRECT`, `QUASI`, `NO_MASK`), and a free-text description of the entity it refers to. That's the gold standard we'll evaluate against in Notebook 02.


In [3]:
# Make src/ importable from the notebook
import sys
sys.path.insert(0, "../src")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
import random
import textwrap

import warnings
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

from anonymisation.data import load_tab, get_dataset_summary

random.seed(42)

print("Loading TAB from HuggingFace (cached after first run)...")
dataset = load_tab()
summary = get_dataset_summary(dataset)

for split, n in summary["splits"].items():
    print(f"  {split:12s}: {n:,} documents")
print(f"  {'TOTAL':12s}: {sum(summary['splits'].values()):,} documents")


Loading TAB from HuggingFace (cached after first run)...


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

  train       : 1,112 documents
  validation  : 541 documents
  test        : 555 documents
  TOTAL       : 2,208 documents


## What kinds of identifiers live in legal text?

Three things to look for:

1. **Entity types** — `PERSON`, `LOC`, `ORG`, `DATETIME`, `QUANTITY`, `CODE`, `DEM` (demographics), `MISC`.
2. **Identifier types** — the *role* the entity plays, which determines whether it needs masking.
3. **The ratio between the two** — how many of these mentions are actually risky.


In [4]:
print("ENTITY TYPES — what kind of information is being flagged?")
print("=" * 60)
for etype, count in summary["entity_types"].most_common():
    bar = "█" * (count // 1000)
    print(f"  {etype:10s} {count:6,}  {bar}")

print()
print("IDENTIFIER ROLES — how sensitive is each mention?")
print("=" * 60)
for itype, count in summary["identifier_types"].most_common():
    print(f"  {itype:10s} {count:6,}")

print("""
Identifier role guide:
  DIRECT   alone, this could identify someone (full name, ID number)
  QUASI    harmless alone, but identifying in combination — DOB, city, role
  NO_MASK  present in the document, but does NOT need masking
""")


ENTITY TYPES — what kind of information is being flagged?
  DATETIME   53,668  █████████████████████████████████████████████████████
  ORG        40,695  ████████████████████████████████████████
  PERSON     24,322  ████████████████████████
  LOC         9,982  █████████
  DEM         8,683  ████████
  MISC        7,044  ███████
  CODE        6,471  ██████
  QUANTITY    4,141  ████

IDENTIFIER ROLES — how sensitive is each mention?
  QUASI      98,244
  NO_MASK    50,023
  DIRECT      6,739

Identifier role guide:
  DIRECT   alone, this could identify someone (full name, ID number)
  QUASI    harmless alone, but identifying in combination — DOB, city, role
  NO_MASK  present in the document, but does NOT need masking



### What this already tells us

`DATETIME` is the most common entity type — there are more dates in TAB than there are people. And most of them are **QUASI**, not DIRECT. Same with `LOC` and `DEM`.

This is the first hint at why the naive answer ("just remove the names") fails. A redacted document with every PERSON masked but the dates, locations, and demographics intact is *not* anonymised — it's a much smaller mosaic that an attacker can still solve.


## Concrete examples — what does each label look like?

It's easy to look at a label like `DEM` and skip past it. Pulling actual examples out of the corpus makes the problem feel real.


In [5]:
buckets = defaultdict(list)
for split in dataset:
    for doc in dataset[split]:
        for m in doc["entity_mentions"]:
            buckets[(m["entity_type"], m["identifier_type"])].append(m["span_text"].strip())

ENTITY_TYPES = [e for e, _ in summary["entity_types"].most_common()]
ID_TYPES = ["DIRECT", "QUASI", "NO_MASK"]
N = 3

print("EXAMPLES BY ENTITY TYPE × IDENTIFIER ROLE")
print("=" * 60)
for etype in ENTITY_TYPES:
    print(f"\n── {etype} ──")
    for itype in ID_TYPES:
        examples = list(dict.fromkeys(buckets.get((etype, itype), [])))
        sample = random.sample(examples, min(N, len(examples)))
        if sample:
            print(f"  {itype:8s} → " + "  |  ".join(f'"{s[:40]}"' for s in sample))
        else:
            print(f"  {itype:8s} → (none)")


EXAMPLES BY ENTITY TYPE × IDENTIFIER ROLE

── DATETIME ──
  DIRECT   → "1955"  |  "6 April 1999"  |  "24 October 1990"
  QUASI    → "8 June 2009"  |  "11 August 1999"  |  "5 August 1998"
  NO_MASK  → "December 1992"  |  "three and a half years"  |  "13 August 1996"

── ORG ──
  DIRECT   → "Vereinigung Bildender Künstler"  |  "Axel Springer AG"  |  "Ruvhten Sijte"
  QUASI    → "Midyat Assize Court"  |  "Christian Trade Union"  |  "Greece"
  NO_MASK  → "State Security Forces"  |  "kommunstyrelsen"  |  "Legal Affairs"

── PERSON ──
  DIRECT   → "Mr Karl-Heinz Bergmann"  |  "Ünal"  |  "Mr Jean-Marie Colombani"
  QUASI    → "Mr M.B. Elmer"  |  "Captain J"  |  "Mrs Muñoz Díaz"
  NO_MASK  → "Ms D.S."  |  "Mr Walsh"  |  "W.S."

── LOC ──
  DIRECT   → "33, 28th October Street, Kyrenia (northe"
  QUASI    → "Izmir"  |  "Wrexham"  |  "Morawica"
  NO_MASK  → "Amsterdam"  |  "Australia"  |  "New York"

── DEM ──
  DIRECT   → "cerebrovascular haemorrhagic shock"
  QUASI    → "overdose of anti-depres

Notice the patterns that are about to make the model's life difficult:

- **CODE** — case file numbers like `Application no. 12345/67`. These are pure DIRECT identifiers but they are *not entities* in the OntoNotes label set that spaCy was trained on. There is no off-the-shelf NER model that knows this is sensitive.
- **DEM** — demographics. `Hungarian`, `Roma`, `pensioner`, `mother of three`. Some of these are NORP (nationality/religion) labels in spaCy, but most are not.
- **QUANTITY** is a flood. Every "ten years", "five months", "twice", "approximately 200" is tagged. Most are NO_MASK. A few (sentence lengths, ages) are QUASI. Telling them apart is non-trivial.


## A single document, end-to-end

To make this concrete, let's pick one document and look at every annotated entity in order.


In [6]:
docs_with_many = sorted(
    dataset["train"], key=lambda d: len(d["entity_mentions"]), reverse=True
)
doc = docs_with_many[2]  # top-3 avoids extreme outliers
text = doc["text"]
mentions = sorted(doc["entity_mentions"], key=lambda m: m["start_offset"])

print(f"Document: {doc['doc_id']}")
print(f"Length: {len(text):,} chars   |   Entity mentions: {len(mentions)}")
print("=" * 60)
print(f"  {'#':<4} {'TYPE':<10} {'ROLE':<8} {'TEXT'}")
for i, m in enumerate(mentions[:30], 1):
    span = m["span_text"].strip().replace("\n", " ")[:50]
    print(f"  {i:<4} {m['entity_type']:<10} {m['identifier_type']:<8} \"{span}\"")
if len(mentions) > 30:
    print(f"  ... and {len(mentions) - 30} more")

print("\n── EXCERPT ──")
print(textwrap.fill(text[:800], width=80, initial_indent="  ", subsequent_indent="  "))


Document: 001-145700
Length: 22,804 chars   |   Entity mentions: 357
  #    TYPE       ROLE     TEXT
  1    CODE       DIRECT   "48311/10"
  2    ORG        NO_MASK  "Federal Republic of Germany"
  3    MISC       QUASI    "public limited company"
  4    DEM        NO_MASK  "German"
  5    ORG        DIRECT   "Axel Springer AG"
  6    DATETIME   QUASI    "19 August 2010"
  7    PERSON     QUASI    "Mr U. Börger"
  8    LOC        QUASI    "Hamburg"
  9    ORG        NO_MASK  "German Government"
  10   ORG        NO_MASK  "Government"
  11   PERSON     QUASI    "Ms K. Behr"
  12   PERSON     QUASI    "Mr H.-J. Behrens"
  13   ORG        NO_MASK  "Federal Ministry of Justice"
  14   DATETIME   QUASI    "28 March 2012"
  15   ORG        NO_MASK  "Government"
  16   ORG        NO_MASK  "Media Legal Defence Initiative"
  17   MISC       QUASI    "public limited company"
  18   LOC        QUASI    "Hamburg"
  19   MISC       QUASI    "newspaper"
  20   ORG        QUASI    "Bild"
  21   DATET

## A first taste of the mosaic effect

Now imagine an adversary who wants to re-identify the subject of this document.

They cannot see the names — those are the DIRECT identifiers. But they *can* see every QUASI mention: the dates, the locations, the demographic descriptors, the quantities. If those facts together are unique within the corpus, the document has effectively re-identified its subject.

Below we build a quick **k-anonymity table**: for every document, we hash the bag of its QUASI mentions into a "fingerprint", then count how many other documents share the same fingerprint.

A document with **k = 1** has a fingerprint shared by no other document. Even with every name masked, it is uniquely identifiable.


In [7]:
from anonymisation.mosaic import k_anonymity_table

ka = k_anonymity_table(list(dataset["test"]))

print(f"Documents with QUASI fingerprint: {len(ka):,}")
print()
print("Distribution of k (smaller k = more re-identifiable):")
print("=" * 50)
for k_val in [1, 2, 3, 4, 5, 10]:
    count = (ka["k"] <= k_val).sum()
    pct = count / len(ka) * 100
    bar = "█" * int(pct / 2)
    print(f"  k ≤ {k_val:<3d}  {count:4,d}  ({pct:5.1f}%)  {bar}")


Documents with QUASI fingerprint: 127

Distribution of k (smaller k = more re-identifiable):
  k ≤ 1     127  (100.0%)  ██████████████████████████████████████████████████
  k ≤ 2     127  (100.0%)  ██████████████████████████████████████████████████
  k ≤ 3     127  (100.0%)  ██████████████████████████████████████████████████
  k ≤ 4     127  (100.0%)  ██████████████████████████████████████████████████
  k ≤ 5     127  (100.0%)  ██████████████████████████████████████████████████
  k ≤ 10    127  (100.0%)  ██████████████████████████████████████████████████


**This is the headline finding for Phase 1.** A large share of TAB test documents are *uniquely identifiable from their QUASI fingerprint alone* — meaning even a perfect PERSON-redactor (which we don't have) wouldn't be enough.

Notebook `03_mosaic_effect` digs into this in detail. For now, hold onto the fact that:

> NER on its own is *necessary but not sufficient* for anonymisation. The next notebook measures how big the NER gap is; the third measures how much of the residual risk is mosaic-shaped rather than name-shaped.


## Where annotators disagreed

A last EDA touch that matters for the framing in Phase 2: TAB's annotators disagreed on a non-trivial share of mentions. That sets the *ceiling* on what a model can achieve — if humans can't agree whether a date is QUASI or NO_MASK, no model will get them all "right" by either annotator's standard.


In [8]:
entity_labels = defaultdict(list)
for split in dataset:
    for doc in dataset[split]:
        for m in doc["entity_mentions"]:
            entity_labels[m["entity_id"]].append(m)

disagreements = {
    eid: ms for eid, ms in entity_labels.items()
    if len({m["identifier_type"] for m in ms}) > 1
}
print(f"Total unique entities:        {len(entity_labels):,}")
print(f"Entities with disagreement:   {len(disagreements):,}")
print(f"Disagreement rate:            {len(disagreements) / len(entity_labels) * 100:.1f}%")


Total unique entities:        108,151
Entities with disagreement:   937
Disagreement rate:            0.9%


## Recap

| Finding | Why it matters for the law firm |
|---|---|
| TAB has 8 entity types; most mentions are **QUASI**, not DIRECT | Stripping names alone leaves the bulk of the risk untouched |
| **CODE** (case numbers) is a TAB-specific category | Off-the-shelf NER has no equivalent label — it's invisible |
| ~~%~~ of TAB test documents have a unique QUASI fingerprint | Mosaic re-identification is real, even after redaction |
| Annotators themselves disagreed on a chunky minority of cases | Don't expect any model to hit 100% — measure against a sensible ceiling |

Onward to **`02_baseline_evaluation.ipynb`** — where we run spaCy's strongest off-the-shelf model against TAB and measure the gap.
